This notebook takes the output of the ortho creation step & precomputes wald

In [3]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
import pandas as pd

2025-11-19 11:24:09.886048: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-19 11:24:10.635211: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/code-server/4.91.1/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

Set up the cluster

In [4]:
local=False
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=8,#cores per slurm job
        memory="128G",#memory per slurm job
        processes=4,#dask workers per slurm job
        job_extra_directives=["-p ycga", 
            f"--job-name=simclust_worker",
            f"--time=2:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=8)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

Let's just do a super simple bounded concurrency approach

In [5]:
from pathlib import Path

In [6]:
# in the real version, de_novo_sim will take a pair, path/name on init and never save it.
# relative paths to individual components can be used, saved, assumed. 
DATA_ROOT=Path("/gpfs/gibbs/pi/reilly/tabula_data")
path=DATA_ROOT/"simulated"
name="shendure_calibrated_sim_with_orthos_20251118"

In [7]:
from dask.distributed import Semaphore, as_completed, get_client

In [16]:
ortho_root=path/name/"orthos"
scmpradat_root=path/name/"scMPRA"
# output_root=path/name/"orthos_with_precomputed_wald_erin_numerical_stability_ALL5"
output_root=path/name/"orthos_with_precomputed_wald_erin_numerical_stability_test"
output_root.mkdir(exist_ok=True)

input_ortho_names=[path.name for path in ortho_root.iterdir()]

#Semaphore(max_leases=2, name="wald-precompute")

def precompute_one_wald(input_root, scmpradat_root, name):
    sem = Semaphore(name="wald-precompute")
    with sem:
        client=get_client()
        dat=scm.scMPRA_data.from_parquet(scmpradat_root/Path(name).with_suffix(".scmpra"))
        dat.ortho_filter()
        ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
        ortho_oi.training_data=dat
        ortho_oi.precompute_wald(client)
        return ortho_oi


        
# futures = [client.submit(precompute_one_wald,) for name_oi in input_ortho_names]

In [15]:
ortho_oi_0=scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name=input_ortho_names[4])

ortho_oi_0.by_cell_type.model['ExEndodermVisceral'].result()

{'llf_total': -157965.67902013613,
 'llfs': array([-157965.67902014]),
 'aic_total': 316241.35804027226,
 'aics': array([316241.35804027]),
 'df_model_total': 155,
 'df': 155,
 'weights': {'x_mu': Intercept                                                           -3.952540
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8174]      2.184018
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8175]      3.449623
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8179]      1.968689
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8192]     -1.605605
                                                                         ...   
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7975]    0.159172
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7978]    3.144878
  C(cre_id, contr.treatment(base='reference'))[T.eef1aP]               9.093677
  C(cre_id, contr.treatment(base='reference'))[T.pgk1P]                7.719981
  C(c

In [12]:
ortho_oi_3=scm.ortho.load(client=client,
                                    path=ortho_root,
                                    name=input_ortho_names[3])

ortho_oi_3.by_cell_type.model['ExEndodermVisceral'].result()

{'llf_total': -156293.70337886037,
 'llfs': array([-156293.70337886]),
 'aic_total': 312897.40675772075,
 'aics': array([312897.40675772]),
 'df_model_total': 155,
 'df': 155,
 'weights': {'x_mu': Intercept                                                           -4.031823
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8174]      2.347827
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8175]      3.517044
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8179]      2.111383
  C(cre_id, contr.treatment(base='reference'))[T.Bend5_chr4_8192]     -1.201280
                                                                         ...   
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7975]    0.126354
  C(cre_id, contr.treatment(base='reference'))[T.Txndc12_chr4_7978]    3.262882
  C(cre_id, contr.treatment(base='reference'))[T.eef1aP]               9.160131
  C(cre_id, contr.treatment(base='reference'))[T.pgk1P]                7.813523
  C(c

In [18]:
# for one replicate

test_particle=precompute_one_wald(input_root=ortho_root,
        scmpradat_root=scmpradat_root,
        name=input_ortho_names[0])

names=[]
errors=[]
for name in test_particle.wald_precomp.by_cell_type:
    names.append(name)
    errors.append(test_particle.wald_precomp.by_cell_type[name].result().debug_msg)
pd.DataFrame({"name":names,"debug":errors}).to_csv("by_cell_types.tsv",sep="\t")

names=[]
errors=[]
for name in test_particle.wald_precomp.by_cre:
    names.append(name)
    errors.append(test_particle.wald_precomp.by_cre[name].result().debug_msg)
pd.DataFrame({"name":names,"debug":errors}).to_csv("by_cre.tsv",sep="\t")


scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [8]:
# for all reps
for i in input_ortho_names:
    test_particle=precompute_one_wald(input_root=ortho_root,
        scmpradat_root=scmpradat_root,
        name=i)

    names=[]
    errors=[]
    for name in test_particle.wald_precomp.by_cell_type:
        names.append(name)
        errors.append(test_particle.wald_precomp.by_cell_type[name].result().debug_msg)
    pd.DataFrame({"name":names,"debug":errors}).to_csv(f"{i}_by_cell_types.tsv",sep="\t")


    names=[]
    errors=[]
    for name in test_particle.wald_precomp.by_cre:
        names.append(name)
        errors.append(test_particle.wald_precomp.by_cre[name].result().debug_msg)
    pd.DataFrame({"name":names,"debug":errors}).to_csv(f"{i}_by_cre.tsv",sep="\t")

    del test_particle



scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 619 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.
scMPRAforge: INFO: Dropped 618 of 2080 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [19]:
client.close()
cluster.close()

In [22]:
!rm ./worker*
!rm *by_cell_types.tsv
!rm *by_cre.tsv

rm: cannot remove './worker*': No such file or directory


In [21]:
!rm -r $output_root